# Meta evaluation against the gateway CLI

Shuffles reference, exact-duplicate, and augmented-query members from the meta zips,
extracts one small batch at a time (so the full archives never sit unpacked),
runs `go run ./cmd/gateway --list ...`, and builds live heatmaps from `events.jsonl`.

Prerequisites: stack up from `infra/docker` (`docker compose up -d`), host gateway
talking to `localhost:4566` / `50051`. Do not reset volumes between batches.

In [ ]:
from __future__ import annotations

import json
import random
import shutil
import subprocess
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import clear_output, display


def repo_root() -> Path:
    """Returns the repository root that contains services/gateway."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "services" / "gateway" / "cmd" / "gateway").is_dir():
            return candidate
    raise FileNotFoundError("run this notebook from the s3-dedup-engine repository")


ROOT = repo_root()
META = ROOT / ".datasets" / "meta"
REFS_DIR = META / "refs_50k_0"
QUERIES_DIR = META / "dev_queries_50k_0"
GROUND_TRUTH = META / "dev_queries_groundtruth.csv"
GATEWAY = ROOT / "services" / "gateway"
WORK = META / ".eval_work"
AGG_EVENTS = WORK / "all_events.jsonl"
AGG_METRICS = WORK / "all_metrics.yaml"

THRESHOLD = 0.60
WORKERS = 10
LIMIT = 200
BATCH_SIZE = 40
DUP_FRACTION = 0.20
SEED = 7
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

print(ROOT)
print(f"limit={LIMIT} batch={BATCH_SIZE} dup_fraction={DUP_FRACTION}")

In [ ]:
@dataclass(frozen=True)
class Member:
    """One zip member tagged with its evaluation cohort."""

    zip_path: Path
    member: str
    cohort: str
    file_name: str
    expected_ref: str


def find_zip(folder: Path) -> Path:
    """Returns the single zip archive inside a meta folder."""
    zips = sorted(folder.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"no zip in {folder}")
    return zips[0]


def list_images(zip_path: Path) -> list[str]:
    """Lists image member names without extracting the archive."""
    with zipfile.ZipFile(zip_path) as archive:
        return [
            name
            for name in archive.namelist()
            if not name.endswith("/") and Path(name).suffix.lower() in IMAGE_EXTS
        ]


def load_ground_truth(path: Path) -> dict[str, str]:
    """Maps query stem to reference stem."""
    mapping: dict[str, str] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        query, ref = line.split(",", 1)
        mapping[query.strip()] = ref.strip()
    return mapping


def stem(name: str) -> str:
    """Returns the file stem for a member or basename."""
    return Path(name).stem


refs_zip = find_zip(REFS_DIR)
queries_zip = find_zip(QUERIES_DIR)
truth = load_ground_truth(GROUND_TRUTH)
ref_members = list_images(refs_zip)
query_members = [m for m in list_images(queries_zip) if stem(m) in truth]
rng = random.Random(SEED)
rng.shuffle(ref_members)
rng.shuffle(query_members)
if LIMIT > 0:
    ref_members = ref_members[:LIMIT]
    query_members = query_members[:LIMIT]

manifest: list[Member] = []
for member in ref_members:
    name = Path(member).name
    manifest.append(Member(refs_zip, member, "ref", name, stem(name)))
dup_count = max(1, int(round(len(ref_members) * DUP_FRACTION))) if ref_members else 0
for member in ref_members[:dup_count]:
    src = Path(member).name
    dup_name = f"{Path(src).stem}_dup{Path(src).suffix}"
    manifest.append(Member(refs_zip, member, "exact_dup", dup_name, stem(src)))
for member in query_members:
    name = Path(member).name
    manifest.append(Member(queries_zip, member, "augmented", name, truth[stem(name)]))

rng.shuffle(manifest)
by_file = {item.file_name: item for item in manifest}
print(
    f"manifest={len(manifest)} refs={len(ref_members)} "
    f"dups={dup_count} augs={len(query_members)}"
)

In [ ]:
OUTCOMES = [
    "inserted",
    "hash_duplicate",
    "semantic_duplicate",
    "replaced",
    "failed",
]
COHORTS = ["ref", "exact_dup", "augmented"]
LABELS = [
    "correct_dedup",
    "wrong_dedup",
    "correct_substitute",
    "wrong_substitute",
    "miss",
    "failed",
]


def classify(event: dict, item: Member) -> str:
    """Labels one event against the expected reference stem."""
    outcome = event.get("outcome", "failed")
    existing = stem(event.get("existing_image_key") or "")
    expected = item.expected_ref
    if outcome == "failed":
        return "failed"
    if outcome in {"hash_duplicate", "semantic_duplicate"}:
        return "correct_dedup" if existing == expected else "wrong_dedup"
    if outcome == "replaced":
        return "correct_substitute" if existing == expected else "wrong_substitute"
    return "miss"


def read_jsonl(path: Path) -> list[dict]:
    """Loads every JSON object from a JSONL file."""
    if not path.is_file():
        return []
    rows: list[dict] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def plot_dashboard(events: list[dict], summary: dict | None) -> None:
    """Redraws cohort and correctness heatmaps plus optional FinOps bars."""
    cohort_counts = pd.DataFrame(0, index=COHORTS, columns=OUTCOMES, dtype=int)
    label_counts = pd.DataFrame(
        0, index=["exact_dup", "augmented"], columns=LABELS, dtype=int
    )
    distances: dict[str, list[float]] = defaultdict(list)
    for event in events:
        item = by_file.get(event.get("file", ""))
        if item is None:
            continue
        outcome = event.get("outcome", "failed")
        if outcome in OUTCOMES:
            cohort_counts.loc[item.cohort, outcome] += 1
        if item.cohort in {"exact_dup", "augmented"}:
            label = classify(event, item)
            label_counts.loc[item.cohort, label] += 1
            if outcome == "semantic_duplicate":
                distances[label].append(float(event.get("semantic_distance") or 0.0))

    clear_output(wait=True)
    cols = 3 if summary else 2
    fig, axes = plt.subplots(1, cols, figsize=(6 * cols, 4.5))
    if cols == 2:
        ax0, ax1 = axes
        ax2 = None
    else:
        ax0, ax1, ax2 = axes
    sns.heatmap(cohort_counts, annot=True, fmt="d", cmap="Blues", ax=ax0)
    ax0.set_title(f"stored outcomes ({len(events)} images)")
    sns.heatmap(label_counts, annot=True, fmt="d", cmap="Oranges", ax=ax1)
    ax1.set_title("dup / query correctness")
    if ax2 is not None and summary:
        keys = [
            "uploaded_unique_count",
            "full_hash_duplicate_count",
            "semantic_duplicate_count",
            "replaced_count",
            "failed_count",
        ]
        ax2.bar(
            [k.replace("_count", "") for k in keys], [summary.get(k, 0) for k in keys]
        )
        ax2.set_title("run summary")
        ax2.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    if distances:
        fig2, ax = plt.subplots(figsize=(7, 3.5))
        for label, values in distances.items():
            if values:
                sns.kdeplot(values, ax=ax, label=label, fill=True, alpha=0.3)
        ax.set_title("semantic distance by correctness label")
        ax.legend()
        display(fig2)
        plt.close(fig2)

In [ ]:
def extract_batch(batch: list[Member], batch_dir: Path) -> Path:
    """Extracts one batch and writes the gateway path list."""
    if batch_dir.exists():
        shutil.rmtree(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)
    by_zip: dict[Path, list[Member]] = defaultdict(list)
    for item in batch:
        by_zip[item.zip_path].append(item)
    for zip_path, items in by_zip.items():
        with zipfile.ZipFile(zip_path) as archive:
            for item in items:
                target = batch_dir / item.file_name
                with archive.open(item.member) as src, target.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
    list_path = batch_dir / "paths.txt"
    list_path.write_text(
        "\n".join(str((batch_dir / item.file_name).resolve()) for item in batch) + "\n",
        encoding="utf-8",
    )
    return list_path


def newest_log_dir(before: set[Path]) -> Path:
    """Returns the dedup-log directory created by the latest gateway run."""
    root = GATEWAY / "dedup-log"
    for _ in range(100):
        dirs = (
            {path for path in root.iterdir() if path.is_dir()}
            if root.is_dir()
            else set()
        )
        created = dirs - before
        if created:
            return max(created, key=lambda path: path.stat().st_mtime)
        time.sleep(0.1)
    raise TimeoutError("gateway did not create a dedup-log directory")


def run_gateway(list_path: Path) -> Path:
    """Runs the host gateway against one path list and returns its log dir."""
    before = (
        {path for path in (GATEWAY / "dedup-log").iterdir() if path.is_dir()}
        if (GATEWAY / "dedup-log").is_dir()
        else set()
    )
    cmd = [
        "go",
        "run",
        "./cmd/gateway",
        "--list",
        str(list_path.resolve()),
        "--threshold",
        str(THRESHOLD),
        "--workers",
        str(WORKERS),
    ]
    proc = subprocess.Popen(
        cmd,
        cwd=GATEWAY,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    log_dir = newest_log_dir(before)
    events_path = log_dir / "events.jsonl"
    summary_path = log_dir / "summary.json"
    seen = 0
    assert proc.stdout is not None
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line, end="")
        events = read_jsonl(events_path)
        if len(events) != seen:
            seen = len(events)
            summary = None
            if summary_path.is_file():
                summary = json.loads(summary_path.read_text(encoding="utf-8"))
            plot_dashboard(read_jsonl(AGG_EVENTS) + events, summary)
        else:
            time.sleep(0.5)
    rest = proc.stdout.read()
    if rest:
        print(rest, end="")
    if proc.returncode not in (0, 1):
        raise RuntimeError(f"gateway exited with {proc.returncode}")
    return log_dir


WORK.mkdir(parents=True, exist_ok=True)
AGG_EVENTS.write_text("", encoding="utf-8")
metrics_acc: Counter[str] = Counter()
all_events: list[dict] = []

for start in range(0, len(manifest), BATCH_SIZE):
    batch = manifest[start : start + BATCH_SIZE]
    batch_dir = WORK / f"batch_{start:05d}"
    print(f"batch {start // BATCH_SIZE + 1}: {len(batch)} images")
    list_path = extract_batch(batch, batch_dir)
    try:
        log_dir = run_gateway(list_path)
        batch_events = read_jsonl(log_dir / "events.jsonl")
        with AGG_EVENTS.open("a", encoding="utf-8") as handle:
            for event in batch_events:
                handle.write(json.dumps(event) + "\n")
                all_events.append(event)
        metrics_path = log_dir / "metrics.yaml"
        if metrics_path.is_file():
            snap = yaml.safe_load(metrics_path.read_text(encoding="utf-8")) or {}
            for key, value in snap.items():
                if (
                    isinstance(value, (int, float))
                    and not key.endswith("percent")
                    and key != "elapsed_seconds"
                ):
                    metrics_acc[key] += value
                elif key == "elapsed_seconds":
                    metrics_acc[key] += value
        plot_dashboard(all_events, dict(metrics_acc))
    finally:
        shutil.rmtree(batch_dir, ignore_errors=True)

AGG_METRICS.write_text(
    yaml.safe_dump(dict(metrics_acc), sort_keys=False), encoding="utf-8"
)
print(f"done images={len(all_events)} metrics={AGG_METRICS}")

In [ ]:
final_summary = yaml.safe_load(AGG_METRICS.read_text(encoding="utf-8")) or {}
plot_dashboard(read_jsonl(AGG_EVENTS), final_summary)
pd.Series(final_summary).to_frame("value")